### **👩‍🏫 👩🏿‍🏫 What You’ll learn**

- Gain an understanding of the basics of web scraping using Python libraries.
- Learn how to use the `requests` library to send HTTP requests and receive responses.
- Understand how to use `BeautifulSoup` to parse HTML content and extract data.
- Acquire skills in extracting specific information like movie ratings and details using HTML tags.
- Learn how to manipulate and store extracted data using `Pandas`, a powerful data analysis tool in Python.

---

### **🛠️ What you will create**

- A Python script to scrape the GitHub Topics webpage.
- Utilizing `BeautifulSoup` to parse the saved HTML content of the GitHub Topics page.
- A dictionary into a `pandas` `DataFrame` for better data manipulation and analysis.

---

### **Web Scraping:**

- use this website: [Github/topics](https://github.com/topics)
- Write a Python script using the `requests` library to fetch the HTML content of the chosen website.
- Print the status code of the response to ensure the request was successful using `.status_code`, it should be `200`.
- Print the first 100 characters of the HTML content to verify the response.
- Save the HTML content to a file named `webpage.html`. Ensure you handle the text encoding correctly.
- Use `BeautifulSoup` to parse the saved HTML content.
- Identify two distinct pieces of information on the webpage to extract (e.g., titles of the topics and their descriptions).
- Write code to extract these pieces of information. Ensure you identify the correct HTML tags and classes used for these elements on the webpage.
- Print the length and content of each extracted list to verify the extraction process.
- Create a Python dictionary to structure the extracted data, with keys representing the type of information (e.g., `‘title’` and `‘description’`).
- Convert this dictionary into a `pandas` `DataFrame`.
- Print the `DataFrame` to confirm its structure and contents.


In [1]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://github.com/topics"
HTML_PATH = Path("webpage.html")
HEADERS = {"User-Agent": "Mozilla/5.0 (DI-Bootcamp web scraping exercise)"}

In [2]:
response = requests.get(URL, headers=HEADERS, timeout=15)
print("Status code:", response.status_code)  # expect 200
response.raise_for_status()  # stop here with a clear error if the request failed

Status code: 200


In [3]:
print(response.text[:100])



<!DOCTYPE html>
<html
  lang="en"
  
  data-color-mode="auto" data-light-theme="light" data-dark-t


In [4]:
# Explicit UTF-8 on write *and* read, so non-ASCII characters (e.g. "Node.js ·", emojis)
response.encoding = "utf-8"
HTML_PATH.write_text(response.text, encoding="utf-8")
print(f"Saved {HTML_PATH} ({HTML_PATH.stat().st_size:,} bytes)")

Saved webpage.html (197,451 bytes)


In [5]:
soup = BeautifulSoup(HTML_PATH.read_text(encoding="utf-8"), "html.parser")

In [6]:
def extract_topics(soup: BeautifulSoup) -> tuple[list[str], list[str]]:
    """Return parallel lists of topic titles and descriptions.

    Each topic is an <a href="/topics/..."> card containing a title <p class="f3">
    and a description <p class="f5">. Reading both from the same card keeps them
    aligned; deduplicating by href removes topics shown both as featured and in the list.
    """
    titles, descriptions, seen = [], [], set()
    for card in soup.select('a[href^="/topics/"]'):
        title_tag = card.select_one("p.f3")
        desc_tag = card.select_one("p.f5")
        if title_tag is None or card["href"] in seen:
            continue  # not a topic card (e.g. nav link) or a duplicate
        seen.add(card["href"])
        titles.append(title_tag.get_text(strip=True))
        descriptions.append(desc_tag.get_text(strip=True) if desc_tag else "")
    return titles, descriptions


titles, descriptions = extract_topics(soup)

In [7]:
print(f"Titles ({len(titles)}):", titles)
print(f"\nDescriptions ({len(descriptions)}):")
for description in descriptions:
    print("-", description)

Titles (16): ['npm', 'Python', 'React', 'Awesome Lists', 'Chrome', 'Code quality', 'Compiler', 'CSS', 'Database', 'Front end', 'JavaScript', 'Node.js', 'Project management', 'React Native', 'Scala', 'TypeScript']

Descriptions (16):
- npm is a package manager for JavaScript included with Node.js.
- Python is a dynamically typed programming language.
- React is an open source JavaScript library used for designing user interfaces.
- An awesome list is a list of awesome things curated by the community.
- Chrome is a web browser from the tech company Google.
- Automate your code review with style, quality, security, and test‑coverage checks when you need them.
- Compilers are software that translate higher-level programming languages to lower-level languages (e.g. machine code).
- Cascading Style Sheets (CSS) is a language used most often to style and improve upon the appearance of views.
- A database is a structured set of data held in a computer, usually a server.
- Front end is the prog

In [8]:
topics_dict = {"title": titles, "description": descriptions}
topics_df = pd.DataFrame(topics_dict)
print(topics_df.shape)
topics_df

(16, 2)


,title,description
0,npm,npm is a package manager for JavaScript includ...
1,Python,Python is a dynamically typed programming lang...
2,React,React is an open source JavaScript library use...
3,Awesome Lists,An awesome list is a list of awesome things cu...
4,Chrome,Chrome is a web browser from the tech company ...
5,Code quality,"Automate your code review with style, quality,..."
6,Compiler,Compilers are software that translate higher-l...
7,CSS,Cascading Style Sheets (CSS) is a language use...
8,Database,A database is a structured set of data held in...
9,Front end,Front end is the programming and layout that p...


## Summary

- Fetched `github.com/topics` with `requests` (status 200) and saved the HTML to `webpage.html` as UTF-8.
- Parsed the saved file with BeautifulSoup. Each topic is an `<a href="/topics/...">` card with a title (`p.f3`) and a description (`p.f5`).
- Reading both fields from the same card keeps titles and descriptions aligned. Deduplicating by link removes topics that appear both as "featured" and in the main list.
- Built a DataFrame with one row per topic (`title`, `description`).
- Limitation: the page shows only the first batch of topics without login or JavaScript ("Load more" is dynamic), so this is a sample, not every GitHub topic.
